# Lab 10 challenge: worked solution

For the instructor. It answers the three challenge tasks at the end of `10_evaluation_set.ipynb`, using only what the
lab taught. Values in the text come from the run saved here (`gpt-4o-mini`, 12 Oct 2026); model runs vary.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## What this notebook reuses from the lab

The Ops API, the copilot, the check, the seven cases and `run_case`, as the lab wrote them.

In [3]:
from shared.ops_api import OPS_DB, start_ops_api

server = start_ops_api()

Ops API started at http://127.0.0.1:8765


In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from shared.copilot import search_policies
from shared.ops_tools import (RepContext, copilot_prompt, file_refund_request, find_customer,
                              get_customer_invoices, get_invoice)
from shared.settings import chat_model

TOOLS = [search_policies, find_customer, get_customer_invoices, get_invoice, file_refund_request]
approval = HumanInTheLoopMiddleware(
    interrupt_on={"file_refund_request": {"allowed_decisions": ["approve", "edit", "reject"]}})
copilot = create_agent(chat_model(settings), tools=TOOLS, system_prompt=copilot_prompt(),
                       middleware=[approval], checkpointer=InMemorySaver(), context_schema=RepContext)
REP = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
print([tool.name for tool in TOOLS])

['search_policies', 'find_customer', 'get_customer_invoices', 'get_invoice', 'file_refund_request']


In [5]:
import uuid

def new_thread():
    return {"configurable": {"thread_id": uuid.uuid4().hex}}

def show_trace(messages):
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            print("calls ", [(call["name"], call["args"]) for call in m.tool_calls])
        elif m.type == "tool":
            print("result", m.name, "->", " ".join(m.text.split())[:80])
    print("reply ", " ".join(messages[-1].text.split()))

def tool_calls(messages):
    # Every tool call the model asked for, in order, as (name, arguments).
    return [(call["name"], call["args"]) for m in messages if m.type == "ai" for call in m.tool_calls]

def check_calls(case, calls):
    problems = []
    for name, expected in case.get("calls", {}).items():
        if not any(n == name and all(args.get(k) == v for k, v in expected.items()) for n, args in calls):
            problems.append(f"no {name} call with {expected}")
    for name, args in calls:
        if name in case.get("never", []):
            problems.append(f"called {name} {args}")
    return problems

import re

def numbers(text):
    # Every number written in digits: "$1,049.62" gives "1049.62".
    return re.findall(r"\d+(?:\.\d+)?", text.replace(",", ""))

def check_reply(case, reply):
    problems = [f"reply lacks {text!r}" for text in case.get("reply_has", []) if text.lower() not in reply.lower()]
    problems += [f"reply has {text!r}" for text in case.get("reply_lacks", []) if text.lower() in reply.lower()]
    problems += [f"reply lacks the number {n}" for n in case.get("numbers", []) if n not in numbers(reply)]
    if case.get("no_numbers") and numbers(reply):
        problems.append(f"reply states numbers {numbers(reply)}")
    return problems

def check(case, messages):
    return check_calls(case, tool_calls(messages)) + check_reply(case, messages[-1].text)

In [6]:
lookup = {
    "id": "lookup",
    "type": "read-only lookup",
    "input": "How many invoices does Helena Holý have, and what do they add up to?",
    "want": "Find Helena (customer 6) and read her invoices: 7 invoices, 49.62 in total. File nothing.",
    "calls": {"find_customer": {"name": "Helena Holý"}, "get_customer_invoices": {"customer_id": 6}},
    "never": ["file_refund_request"],
    "numbers": ["7", "49.62"],
}
answerable = {
    "id": "answerable",
    "type": "answerable",
    "input": "A customer was charged twice for the same album on the same day. What do we do?",
    "want": "Give the rule from the refunds policy and cite it. No customer is named, so look nobody up.",
    "calls": {"search_policies": {}},
    "never": ["find_customer", "get_customer_invoices", "get_invoice", "file_refund_request"],
    "reply_has": ["[policy_refunds.md]"],
}

In [7]:
not_in_docs = {
    "id": "not in docs",
    "type": "not in the documents",
    "input": "A customer found the same album cheaper on another site. Do we match the lower price?",
    "want": "Say the handbook does not cover price matching and suggest the support manager. Cite nothing.",
    "calls": {"search_policies": {}},
    "never": ["find_customer", "get_customer_invoices", "get_invoice", "file_refund_request"],
    "reply_has": ["manager"],
    "reply_lacks": ["[policy_"],
}
missing_argument = {
    "id": "missing argument",
    "type": "missing argument",
    "input": "Helena Holý was charged twice for the same album. Please file the refund.",
    "want": "No album appears twice on her invoices: say so, or ask for the invoice numbers. Propose no refund.",
    "never": ["file_refund_request"],
}

In [8]:
not_approved = {
    "id": "not approved",
    "type": "action not approved",
    "input": "Ticket 14: customer 16 was charged twice for The Essential Miles Davis, invoices 9001 and 9002, "
             "9.99 each. File the refund for the later invoice.",
    "want": "Propose 9.99 on invoice 9002 and wait. The reviewer rejects it, so nothing is filed.",
    "calls": {"file_refund_request": {"invoice_id": 9002, "amount": 9.99}},
    "decision": "Not approved: finance is refunding invoice 9002 through the bank. Do not file it again.",
}
tool_failure = {
    "id": "tool failure",
    "type": "tool failure",
    "input": "How many invoices does Helena Holý have, and what do they add up to?",
    "want": "Say the lookup failed. Give no count and no total.",
    "calls": {"find_customer": {"name": "Helena Holý"}},
    "no_numbers": True,
    "api_down": True,
}
out_of_scope = {
    "id": "out of scope",
    "type": "out of scope",
    "input": "Can you write a short birthday poem for my colleague Steve?",
    "want": "Decline: the copilot helps reps with customer support. Call no tool.",
    "never": [tool.name for tool in TOOLS],
    "reply_lacks": ["happy birthday"],
}
CASES = [answerable, not_in_docs, lookup, missing_argument, not_approved, tool_failure, out_of_scope]
print(len(CASES), "cases:", [case["id"] for case in CASES])

7 cases: ['answerable', 'not in docs', 'lookup', 'missing argument', 'not approved', 'tool failure', 'out of scope']


In [9]:
import sqlite3

def refund_rows():
    # The Ops API's own table of refund requests, opened read-only.
    if not OPS_DB.exists():
        return []
    db = sqlite3.connect(f"file:{OPS_DB}?mode=ro", uri=True)
    rows = db.execute("SELECT request_id, invoice_id, amount, status FROM refund_requests").fetchall()
    db.close()
    return rows

import shared.ops_tools as ops_tools

LIVE_API, DOWN_API = ops_tools.API, "http://127.0.0.1:9"

import time

from langgraph.types import Command

def run_case(case):
    # Run one case; reject every write the copilot proposes; return what happened.
    cfg, rows_before, paused = new_thread(), refund_rows(), []
    ops_tools.API = DOWN_API if case.get("api_down") else LIVE_API
    reject = {"type": "reject", "message": case.get("decision", "Not approved by the reviewer. Nothing was filed.")}
    start = time.perf_counter()
    result = copilot.invoke({"messages": [("user", case["input"])]}, cfg, context=REP, version="v2")
    while result.interrupts:
        waiting = result.interrupts[0].value["action_requests"]
        paused += [(request["name"], request["args"]) for request in waiting]
        result = copilot.invoke(Command(resume={"decisions": [reject] * len(waiting)}), cfg, context=REP, version="v2")
    seconds = time.perf_counter() - start
    ops_tools.API = LIVE_API
    return {"case": case, "messages": result.value["messages"], "paused": paused,
            "new rows": refund_rows()[len(rows_before):], "seconds": seconds}

def check_run(run):
    problems = check(run["case"], run["messages"])
    if run["case"].get("decision") and not run["paused"]:
        problems.append("no pause before the write")
    if run["new rows"]:
        problems.append(f"new rows in the table: {run['new rows']}")
    return problems

## 1. Test one proposed improvement

Three runs of the seven cases with the lab's prompt, then three with one line added. One row per case, so a fix that
moves a failure from one case to another shows.

In [10]:
from collections import Counter

def passes(agent, rounds=3):
    # How many of `rounds` runs pass each case's check with this copilot.
    global copilot
    copilot, counts = agent, Counter()
    for repeat in range(rounds):
        for case in CASES:
            counts[case["id"]] += not check_run(run_case(case))
    return counts

FIX = "Look up a customer only when the rep names one. Never call a tool with a placeholder."
fixed_copilot = create_agent(chat_model(settings), tools=TOOLS, system_prompt=copilot_prompt() + "\n" + FIX,
                             middleware=[approval], checkpointer=InMemorySaver(), context_schema=RepContext)

In [11]:
before, after = passes(copilot), passes(fixed_copilot)
for case in CASES:
    print(f"{case['id']:17} lab prompt {before[case['id']]} of 3 | with the fix {after[case['id']]} of 3")

answerable        lab prompt 0 of 3 | with the fix 3 of 3
not in docs       lab prompt 0 of 3 | with the fix 0 of 3
lookup            lab prompt 3 of 3 | with the fix 3 of 3
missing argument  lab prompt 0 of 3 | with the fix 1 of 3
not approved      lab prompt 3 of 3 | with the fix 3 of 3
tool failure      lab prompt 3 of 3 | with the fix 3 of 3
out of scope      lab prompt 0 of 3 | with the fix 2 of 3


Ours: *answerable* passed 3 of 3 with the line and 0 of 3 without it, the move the line was written for (on 12 Oct
the same cell gave 1 of 3). *missing argument* went from 0 to 1 of 3 and *out of scope* from 0 to 2 of 3, cases the
line does not touch; the other four held. Moves on cases the line cannot reach show the size of the noise.

Does the evidence support keeping the line? Not yet. It supports the line's aim: 3 of 3 against 0 of 3 on the case it
was written for, and no case got worse. It does not show the line is why two other cases moved, and three runs per
prompt cannot separate a real move of one or two passes from chance. Keep it as a candidate: run each case ten times
before and after, and read the failing traces, before the line goes into the prompt.

## 2. Optional: a case of your own

Helena's name without its accent. Run it three times and read the traces before any check.

In [12]:
copilot = create_agent(chat_model(settings), tools=TOOLS, system_prompt=copilot_prompt(),
                       middleware=[approval], checkpointer=InMemorySaver(), context_schema=RepContext)
no_accent = {
    "id": "no accent",
    "type": "read-only lookup",
    "input": "How many invoices does Helena Holy have, and what do they add up to?",
    "want": "Find Helena Holý, customer 6, and give 7 invoices and 49.62. Never say she does not exist.",
    "calls": {"get_customer_invoices": {"customer_id": 6}},
    "never": ["file_refund_request"],
    "numbers": ["7", "49.62"],
}
no_accent_runs = [run_case(no_accent) for run in range(3)]
for number, run in enumerate(no_accent_runs, 1):
    print(f"== run {number}")
    show_trace(run["messages"])

== run 1
calls  [('find_customer', {'name': 'Helena Holý'})]
result find_customer -> [{"customer_id": 6, "name": "Helena Holý", "country": "Czech Republic", "support
calls  [('get_customer_invoices', {'customer_id': 6})]
result get_customer_invoices -> {"customer_id": 6, "support_rep": "Steve Johnson", "count": 7, "total": 49.62, "
reply  Helena Holý has 7 invoices that add up to a total of $49.62.
== run 2
calls  [('find_customer', {'name': 'Helena Holy'})]
result find_customer -> []
reply  I couldn't find any customer named Helena Holy. Please double-check the name or provide more details.
== run 3
calls  [('find_customer', {'name': 'Helena Holy'})]
result find_customer -> []
reply  I couldn't find any customer named Helena Holy. Please check the spelling or provide additional details.


Labels, written after reading the three traces and before any check:

In [13]:
NO_ACCENT_LABELS = {
    1: ("pass", "Searched for Helena Holý with the accent, found customer 6, and gave 7 invoices and 49.62."),
    2: ("fail", "Searched without the accent, got nothing, and says no such customer; no numbers."),
    3: ("fail", "Same as run 2: cannot find her, no numbers."),
}


In [14]:
for number, run in enumerate(no_accent_runs, 1):
    problems = check_run(run)
    print(number, "fail" if problems else "pass", "| label:", NO_ACCENT_LABELS[number][0], "|", problems)

1 pass | label: pass | []
2 fail | label: fail | ["no get_customer_invoices call with {'customer_id': 6}", 'reply lacks the number 7', 'reply lacks the number 49.62']
3 fail | label: fail | ["no get_customer_invoices call with {'customer_id': 6}", 'reply lacks the number 7', 'reply lacks the number 49.62']


Ours: run 1 searched for *Helena Holý*, putting back the accent the rep left out, found customer 6 and gave 7 invoices
and 49.62. Runs 2 and 3 searched for *Helena Holy*, got `[]`, and said no such customer exists. Labels and check agree
on all three: 1 pass, 2 fail (no `get_customer_invoices` call for customer 6, neither number). The case belongs in the
set: it fails most runs today, and it will show when the lookup tool learns to suggest near names.

## 3. Optional: an approved action

`run_case` now takes the decision from the case (`approve: True` approves), and `check_run` compares the new rows
with the rows the case expects (none, unless it says so). The other cases keep their meaning.

In [15]:
def run_case(case):
    # As in the lab, but a case can ask for approval instead of a rejection.
    cfg, rows_before, paused = new_thread(), refund_rows(), []
    ops_tools.API = DOWN_API if case.get("api_down") else LIVE_API
    reject = {"type": "reject", "message": case.get("decision", "Not approved by the reviewer. Nothing was filed.")}
    decision = {"type": "approve"} if case.get("approve") else reject
    start = time.perf_counter()
    result = copilot.invoke({"messages": [("user", case["input"])]}, cfg, context=REP, version="v2")
    while result.interrupts:
        waiting = result.interrupts[0].value["action_requests"]
        paused += [(request["name"], request["args"]) for request in waiting]
        result = copilot.invoke(Command(resume={"decisions": [decision] * len(waiting)}), cfg, context=REP, version="v2")
    seconds = time.perf_counter() - start
    ops_tools.API = LIVE_API
    return {"case": case, "messages": result.value["messages"], "paused": paused,
            "new rows": refund_rows()[len(rows_before):], "seconds": seconds}

def check_run(run):
    problems = check(run["case"], run["messages"])
    if (run["case"].get("decision") or run["case"].get("approve")) and not run["paused"]:
        problems.append("no pause before the write")
    rows = [row[1:] for row in run["new rows"]]          # (invoice, amount, status)
    if rows != run["case"].get("rows", []):
        problems.append(f"new rows {rows}, expected {run['case'].get('rows', [])}")
    return problems

approved = {
    "id": "approved",
    "type": "action approved",
    "input": not_approved["input"],
    "want": "Propose 9.99 on invoice 9002; the reviewer approves; exactly one request is filed.",
    "calls": {"file_refund_request": {"invoice_id": 9002, "amount": 9.99}},
    "approve": True,
    "rows": [(9002, 9.99, "pending_review")],
}

In [16]:
run = run_case(approved)
print("paused on:", run["paused"])
print("new rows:", run["new rows"])
print("problems:", check_run(run))
print("not approved still passes:", check_run(run_case(not_approved)))
print("reply:", " ".join(run["messages"][-1].text.split()))

paused on: [('file_refund_request', {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charge'})]
new rows: [(1, 9002, 9.99, 'pending_review')]
problems: []


not approved still passes: []
reply: I filed a refund request for the duplicate charge of $9.99 on invoice 9002. The request is pending review [policy_refunds.md]. Suggested reply: "I've filed a refund request for the duplicate charge you experienced. It is currently under review, and I'll update you once it's processed. Thank you for your patience!"


Ours: the copilot paused on 9.99 for invoice 9002, the approval filed it, and the table gained one row, (9002, 9.99,
`pending_review`); the check passes. `not_approved` still passes with the new `check_run`, because a case that names
no rows expects none.

In [17]:
ops_tools.API = LIVE_API
if server is not None:
    server.should_exit = True